# Ask A Manager Salary Survey - Data Cleaning Project

### Project Overview

This project focuses on cleaning and preparing the real-world 'salary survey data' collected from the blog [Ask a Manager](https://www.askamanager.org/2021/04/how-much-money-do-you-make-4.html). The dataset contains thousands of self-reported salary records across different industries, job roles, locations, and demographic groups.

The data is raw and contains multiple inconsistencies, such as missing values, varied formats, and free-text inputs. This makes it well-suited for a real-world data cleaning project.

###  Objective  
The primary objective of this project is to transform raw, unstructured survey data into a clean, structured, and analysis-ready dataset.

### Dataset Description  
The dataset includes fields such as job title, industry, annual salary, currency, location, experience level, education, gender, and race.

- Row: ~28000
- Columns: 18

You can download the dataset [here](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/edit?resourcekey=&gid=1625408792#gid=1625408792).

### Data Cleaning Workflow

| Step | Description |
|------|------|
| 1 | Loading and Inspecting Data |
| 2 | Main Data Cleaning Issues |
| 3 | Renaming Columns|
| 4 | Handling Missing values |
| 5 | Converting 'timestamp' column to datatime format|
| 6 |  Parsing 'annual_salary' column |
| 7 |Standardizing country names|
| 8 | Standardizing states names|
| 9 |  Cleaning 'industry' and 'job title' columns|
| 10 | Exporting Cleaned Data |

### 1) Loading and Inspecting Data 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_row = pd.read_csv("datasets/manager_salary_survey_uncleaned.csv")
df_row.head(3)

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


In [3]:
# Shape of data
df_row.shape

(28211, 18)

In [4]:
# Columns
df_row.columns

Index(['Timestamp', 'How old are you?', 'What industry do you work in?',
       'Job title',
       'If your job title needs additional context, please clarify here:',
       'What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)',
       'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
       'Please indicate the currency',
       'If "Other," please indicate the currency here: ',
       'If your income needs additional context, please provide it here:',
       'What country do you work in?',
       'If you're in the U.S., what state do you work in?',
       'What city do you work in?',
       'How many years of professional work experience do you have overall?',
       

In [5]:
df_row.sample(5)

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
9387,4/27/2021 17:50:30,25-34,Computing or Tech,Staff Technical Writer,NaN,"142,000",33000.0,USD,NaN,NaN,U.S.,Texas,Austin,8 - 10 years,8 - 10 years,College degree,Woman,White
25410,5/6/2021 14:47:08,35-44,Computing or Tech,Software Engineer I,NaN,103500,8000.0,USD,NaN,NaN,USA,Oregon,Portland,2 - 4 years,1 year or less,Some college,Man,White
22434,4/30/2021 20:01:30,35-44,Media & Digital,Director of Programming,NaN,150000,22000.0,USD,NaN,NaN,USA,New York,NYC,11 - 20 years,8 - 10 years,College degree,Woman,White
1749,4/27/2021 11:31:15,25-34,Education (Higher Education),Director of Student Engagement,NaN,"50,000",NaN,USD,NaN,NaN,United States,Kansas,Oklahoma City,5-7 years,2 - 4 years,Master's degree,Woman,White
24691,5/5/2021 17:56:35,35-44,Law,Settlement Project Manager,Class Action Settlement Administration,77000,7700.0,USD,NaN,Bonus is once annually in Q1 if Profit goals a...,United States,Washington,Seattle,11 - 20 years,5-7 years,"Professional degree (MD, JD, etc.)",Woman,White


In [6]:
# Data summary
null_summary = pd.DataFrame({
    "dtype"     : df_row.dtypes,
    "nulls"     : df_row.isna().sum(),
    "null_prct"    : (df_row.isna().mean() * 100).round(1),
    "unique_values"  : df_row.nunique(),
})
null_summary

,dtype,nulls,null_prct,unique_values
Timestamp,object,0,0.0,25429
How old are you?,object,0,0.0,7
What industry do you work in?,object,85,0.3,1225
Job title,object,4,0.0,14429
"If your job title needs additional context, please clarify here:",object,20920,74.2,7027
"What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)",object,0,0.0,4330
"How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",float64,7371,26.1,849
Please indicate the currency,object,0,0.0,11
"If ""Other,"" please indicate the currency here:",object,27989,99.2,129
"If your income needs additional context, please provide it here:",object,25158,89.2,2988


### 2) Main Data Cleaning Issues

#### 1. Long Columns names
Column names are very long and need to be shortened for better readability and usability.

#### 2. Missing Values
Several columns contain a high percentage of missing values or sparse data:
- `Additional Job Context` -> High percentage of missing values.
- `Additional monetary compenstation` -> High percentage of missing values.
- `Other Currency` -> Extremely sparse
- `Additional Income Context` -> Extermely sparse
- Some other columns also contain missing values

#### 3. Wrong Data types
Some columns are assigned with wrong datatypes.
- `Time Stamp`, `Annual Salary` are assigned with wrong datatypes.

#### 4. Inconsistent values in few columns
 Columns like `Country`, `State`, `city`, `Job Title`, etc., contain inconsistent data values such as:
- Case inconsitency ("manager", "Manager")
- Different spellings ("USA", "US", "United States")
- Variations and synonyms of some names
  
#### 5. Columns with many unique values
- Columns with 'free text entries' contain too many unique values making analysis and grouping challenging.

### 3) Renaming columns 

In [7]:
# Column Mapping
column_map = {"Timestamp" : "timestamp",
    'How old are you?': "age_group",
    'What industry do you work in?': "industry",
    'Job title': "job_title",
    'If your job title needs additional context, please clarify here:': "job_context",
    df_row.columns[5]: "annual_salary",
    df_row.columns[6]: "other_compensation",
    'Please indicate the currency': "currency",
    'If "Other," please indicate the currency here: ': "other_currency",
    'If your income needs additional context, please provide it here:': "income_context",
    'What country do you work in?': "country",
    "If you're in the U.S., what state do you work in?": "state_us_only",
    'What city do you work in?': "city",
    'How many years of professional work experience do you have overall?': "experience_overall",
    'How many years of professional work experience do you have in your field?': "experience_field",
    'What is your highest level of education completed?': "education",
    "What is your gender?": "gender",
    "What is your race? (Choose all that apply.)": "race"}
                             
df = df_row.rename(columns=column_map)

### 4) Handling Missing values

In [8]:
# Null value percentage in each column
df.isna().mean() * 100

timestamp              0.000000
age_group              0.000000
industry               0.301301
job_title              0.014179
job_context           74.155471
annual_salary          0.000000
other_compensation    26.128106
currency               0.000000
other_currency        99.213073
income_context        89.177980
country                0.003545
state_us_only         17.964624
city                   0.297756
experience_overall     0.000000
experience_field       0.000000
education              0.847187
gender                 0.652228
race                   0.687675
dtype: float64

##### 4.1) Droping columns with high percentage of null values

In [9]:
# 'job_context', 'other_currency' and 'income_context' columns have high percentage of missing values
df.drop(columns=["job_context", "other_currency", "income_context"],inplace=True)

##### 4.2) Imputing missing values in 'other_compensation' with 0

In [10]:
# We are filling with 0 because the column question specifies 'if any', meaning absence implies on additional compensation
df["other_compensation"] = df["other_compensation"].fillna(0)

##### 4.3)  Filling missing values in categorical column with 'Unknown'.

In [11]:
# Columns like 'industry', 'job_title', 'country', 'city' and 'education' contain a small amount of missing values, which 
# we can replace with 'Unknown'.

cat_col = ["industry", "job_title", "country", "city", 'education']

for col in cat_col:
    df[col] = df[col].fillna("Unknown")

##### 4.4) Handling missing values in 'gender' column

In [12]:
# Inspecting unique values in 'gender' column
df["gender"].unique()

array(['Woman', 'Non-binary', 'Man', nan, 'Other or prefer not to answer',
       'Prefer not to answer'], dtype=object)

In [13]:
# Filling missing values with 'Other or prefer not to answer'
df["gender"] = df["gender"].fillna("Other or prefer not to answer")

In [14]:
# Grouping Gender values into three categories: 'Man', 'Woman' and 'Other or prefer not to answer'

df["gender"] = df["gender"].replace({
    "Non-binary": "Other or prefer not to answer",
    "Prefer not to answer": "Other or prefer not to answer"})

# This simplifies analysis by combining less frequent and non-disclosed responses into a single category.

##### 4.5) Handling missing values in 'race' column

In [15]:
# Inspecting unique values in 'race' column
df["race"].unique()

array(['White', 'Hispanic, Latino, or Spanish origin, White',
       'Asian or Asian American, White', 'Asian or Asian American',
       'Another option not listed here or prefer not to answer',
       'Hispanic, Latino, or Spanish origin',
       'Middle Eastern or Northern African',
       'Hispanic, Latino, or Spanish origin, Middle Eastern or Northern African, White',
       'Black or African American', 'Black or African American, White',
       nan,
       'Black or African American, Hispanic, Latino, or Spanish origin, White',
       'Native American or Alaska Native',
       'Native American or Alaska Native, White',
       'Hispanic, Latino, or Spanish origin, Another option not listed here or prefer not to answer',
       'Black or African American, Middle Eastern or Northern African, Native American or Alaska Native, White',
       'White, Another option not listed here or prefer not to answer',
       'Black or African American, Native American or Alaska Native, White',
    

In [16]:
# Filling missing values with 'Another option not listed here or prefer not to answer' in race column

df["race"] = df["race"].fillna('Another option not listed here or prefer not to answer')

In [17]:
df.isna().sum()

timestamp                0
age_group                0
industry                 0
job_title                0
annual_salary            0
other_compensation       0
currency                 0
country                  0
state_us_only         5068
city                     0
experience_overall       0
experience_field         0
education                0
gender                   0
race                     0
dtype: int64

### 5) Converting 'timestamp' column to datatime format

In [18]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

### 6) Parsing 'annual_salary' column

In [19]:
# Unique characters in this column
unique_chars = set(''.join(df["annual_salary"].dropna()))
print(unique_chars)

{'7', '9', '0', ',', '2', '5', '4', '6', '3', '8', '1'}


In [20]:
# Removing ',' and converting 'annual_salary' into int type
df["annual_salary"] = df["annual_salary"].str.replace(",","").astype(int)

### 7) Standardizing country names

##### 7.1) Normalizing texts

In [21]:
# Removing leading\trailing whitespace and converting to lowercase
df["country"] = df["country"].str.strip().str.lower()

##### 7.2) Inspecting unique country vales

In [22]:
# Inspecting unique values to understand the country column
df["country"].sort_values().unique()

array(['$2,175.84/year is deducted for benefits', '1', 'afghanistan',
       'africa', 'america', 'aotearoa new zealand', 'argentina',
       'argentina but my org is in thailand', 'australi', 'australia',
       'australian', 'austria',
       'austria, but i work remotely for a dutch/british company',
       'bangladesh', 'belgium', 'bermuda',
       'bonus based on meeting yearly goals set w/ my supervisor',
       'bosnia and herzegovina', 'brasil', 'brazil', 'britain',
       'bulgaria', 'burma', 'california', 'cambodia', 'can', 'canad',
       'canada', 'canada and usa', 'canada, ottawa, ontario', 'canadw',
       'canadá', 'canda', 'catalonia', 'cayman islands', 'chile', 'china',
       'colombia', 'company in germany. i work from pakistan.', 'congo',
       'contracts', 'costa rica', "cote d'ivoire", 'croatia', 'csnada',
       'cuba', 'currently finance', 'cyprus', 'czech republic', 'czechia',
       'danmark', 'dbfemf', 'denmark', 'ecuador', 'egypt', 'england',
       'englan

##### 7.3) Removing invalid country names

In [23]:
# Replacing invalid country names with 'Unknown' (entries that are clearly not country names > entry with more than 30 characters)

df.loc[df["country"].str.len() > 30, "country"] = "Unknown"

##### 7.4) Normalizing United States variants 

In [24]:
# The column contains 30+ variations referring to the 'United States',including abbreviations, misspelings and other indirect names.

usa_variant = {
    'uniited states','unite states','united  states',  'united sates', 'united sates of america',
       'united stares', 'united state', 'united state of america','united statea', 'united stated', 'united stateds',
       'united statees', 'united states', 'united states is america','united states- puerto rico',
       'united states of america', 'united states of american','united states of americas',
       'united statesp', 'united statew', 'united statss','united stattes', 'united statues', 'united status',
       'united statws', 'united sttes', 'unitedstates','uniteed states', 'unitef stated', 'uniter statez',
       'unites states', 'unitied states','uniyed states', 'uniyes states', 'unted states', 'untied states',
       'us', 'us of a', 'usa', 'usa tomorrow',"usa, but for foreign gov't", 'usa-- virgin islands', 'usaa',
       'usab', 'usat', 'usd', 'uss','🇺🇸', 'the united states', 'the us','america','isa','california',
       'u. s', 'u. s.', 'u.s', 'u.s.', 'u.s.a','u.s.a.', 'u.s>', 'u.sa', 'san francisco',
        'virginia','united states- puerto rico' }

# Replacing United States variants with 'United States';
df["country"] = df["country"].replace({val: "United States" for val in usa_variant})

##### 7.5) Normalizing United Kingdom variants 

In [25]:
# The column contains 30+ variations referring to the 'United Kingdom', including abbreviations, misspelings and other indirect names.

uk_variants = {
    'u.k.', 'u.k. (northern england)', 'uk','u.k',
    'uk (england)', 'uk (northern ireland)', 'uk for u.s. company',
    'uk, remote', 'united kindom','united kingdom', 'united kingdom (england)', 'united kingdom.',
    'united kingdomk','unites kingdom',  'wales','wales (uk)', 'wales (united kingdom)', 'wales, uk',
    'scotland', 'scotland, uk','northern ireland','northern ireland','britain',
    'england', 'england, gb', 'england, uk','england, uk.', 'england, united kingdom', 'england/uk', 'englang',
    'great britain', "london"}


# Replacing United Kingdom variants with a Standard Country name
df["country"] = df["country"].replace({val: "United Kingdom" for val in uk_variants})

In [26]:
# Unique names after normalizing United States and United Kingdom names
df["country"].sort_values().unique()

array(['1', 'United Kingdom', 'United States', 'Unknown', 'afghanistan',
       'africa', 'aotearoa new zealand', 'argentina', 'australi',
       'australia', 'australian', 'austria', 'bangladesh', 'belgium',
       'bermuda', 'bosnia and herzegovina', 'brasil', 'brazil',
       'bulgaria', 'burma', 'cambodia', 'can', 'canad', 'canada',
       'canada and usa', 'canada, ottawa, ontario', 'canadw', 'canadá',
       'canda', 'catalonia', 'cayman islands', 'chile', 'china',
       'colombia', 'congo', 'contracts', 'costa rica', "cote d'ivoire",
       'croatia', 'csnada', 'cuba', 'currently finance', 'cyprus',
       'czech republic', 'czechia', 'danmark', 'dbfemf', 'denmark',
       'ecuador', 'egypt', 'eritrea', 'estonia', 'europe', 'ff',
       'finland', 'france', 'germany', 'ghana', 'global', 'greece',
       'hartford', 'hong kong', 'hong kongkong', 'hong konh', 'hungary',
       'i.s.', 'ibdia', 'india', 'indonesia', 'international', 'ireland',
       'is', 'isle of man', 'israel',

##### 7.6) Normalizing other country names and variants

In [27]:
# Mapping inconsistent country names, misspellings, and alternate representations.
country_map = {
    'aotearoa new zealand': "New Zealand",
    'new zealand aotearoa': "New Zealand",
    'nz':"New Zealand",
    'new zealand':"New Zealand",

    'australi': "Australia",
    'australia':"Australia",
    'australian':"Australia",

    'brasil': 'Brazil',
    'brazil': 'Brazil',

    'burma': "Myanmar",
    'myanmar': "Myanmar",

    'can': "Canada", 
    'canad': "Canada", 
    'canada': "Canada",
    'canada and usa': "Canada",
    'canada, ottawa, ontario': "Canada",
    'canadw': "Canada",
    'canadá': "Canada",
    'canda': "Canada",
    'csnada': "Canada",

    'catalonia': "Spain",
    "spain": "Spain",

    'česká republika': 'Czech Republic',
    'czechia': 'Czech Republic',
    'czech republic': 'Czech Republic',

    'china': "China",
    'mainland china': "China",
        
    'danmark': "Denmark",
    'denmark': "Denmark",

    'hong kong': "Hong Kong",
    'hong kongkong': "Hong Kong",
    'hong konh': "Hong Kong",
    
    'ibdia': "India",
    'india': "India",

    'italia': "Italy",
    'italy': "Italy",
    'italy (south)': "Italy",

    'japan': "Japan",
    'japan, us gov position': "Japan",

    'luxembourg': "Luxembourg",
    'luxemburg': "Luxembourg",

    'mexico': 'Mexico',
    'méxico': "Mexico",

    'nederland':  "Netherlands",
    'netherlands': "Netherlands",
    'nl': "Netherlands",
    'the netherlands': "Netherlands",

    'nigeria': "Nigeria",
    'nigeria + uk': "Nigeria",

    'philippines': 'Philippines',
    'remote (philippines)': 'Philippines',

    'uae': "United Arab Emirates",
    'united arab emirates': "United Arab Emirates",

    "is": "Iceland",
    "i.s.": "Iceland",

    "u.a.": "Ukraine",
    "ua": "Ukraine",
    "ukraine": "ukraine"
}

# Appling mapping to standardize country names
df["country"] = df["country"].replace(country_map)

# Converting country names to title case
df["country"] = df["country"].str.title()

##### 7.7) Identify rare values in Country column

In [28]:
# Identifying rare values (may contain invalid country names)

countries = df["country"].value_counts().reset_index()
countries[countries["count"] == 1]

,country,count
69,Kuwait,1
70,Contracts,1
71,"Jersey, Channel Islands",1
72,Uxz,1
73,Russia,1
74,Serbia,1
75,Currently Finance,1
76,Rwanda,1
77,Cayman Islands,1
78,Trinidad And Tobago,1


In [29]:
# Replacing manually identified invalid country names with 'Unknown'
non_country = {"Contracts", "Global","Hartford", "Uxz", "Currently Finance", 
               "International", "United Y", "Y", "Africa", "Europe", "Na",
               "Ss", "Policy", "Dbfemf", "Loutreland", "Ff", "1", "Remote"}

df.loc[df["country"].isin(non_country), "country"] = "Unknown"

In [30]:
df["country"].unique()

array(['United States', 'United Kingdom', 'Canada', 'Netherlands',
       'Australia', 'Spain', 'Finland', 'France', 'Germany', 'Ireland',
       'India', 'Argentina', 'Denmark', 'Switzerland', 'Bermuda',
       'Malaysia', 'Mexico', 'South Africa', 'Belgium', 'Sweden',
       'Hong Kong', 'Kuwait', 'Norway', 'Sri Lanka', 'Unknown', 'Greece',
       'Japan', 'Austria', 'Brazil', 'Hungary', 'Luxembourg', 'Colombia',
       'New Zealand', 'Trinidad And Tobago', 'Cayman Islands', 'Ukraine',
       'Czech Republic', 'Latvia', 'Puerto Rico', 'Rwanda',
       'United Arab Emirates', 'Bangladesh', 'Romania', 'Serbia',
       'Philippines', 'Russia', 'Poland', 'Turkey', 'Italy',
       'Jersey, Channel Islands', 'China', 'Afghanistan', 'Israel',
       'Iceland', 'Taiwan', 'Cambodia', 'Vietnam', 'Singapore',
       'South Korea', 'Thailand', 'Lithuania', 'Eritrea', 'Indonesia',
       'Cuba', 'Slovenia', "Cote D'Ivoire", 'Somalia', 'Slovakia',
       'Portugal', 'Sierra Leone', 'The Bahamas', 

### 8) Standardizing states names

##### 8.1) Handling Missing values in 'states_us_only'.

In [31]:
# Labeling non-US respondents
df.loc[(df["country"] != "United States") & (df["state_us_only"].isna()), "state_us_only"] = "Outside_US"

In [32]:
# Handling remaning missing values
df["state_us_only"] = df["state_us_only"].fillna("Unknown")

##### 8.2) Filling 'country' with 'United States' where 'state_us_only' indicates a US state (not Outside_US or Unknown)

In [33]:
# Fill 'country' with 'United States' where 'state_us_only' indicates a US state (not Outside_US or Unknown)

mask = ((df["state_us_only"] != "Outside_US") 
       & (df["state_us_only"] != "Unknown")
       &(df["country"] == "Unknown"))

df.loc[mask, "country"] = "United States"

##### 8.3) Spliting multi-state values into lists

In [34]:
df["state_us_only"] = df["state_us_only"].str.split(",")

### 9) Cleaning 'industry' and 'job title' columns 

In [35]:
# 'industry' and 'job_title' columns are free-text fields.We can strip whitespace and casing.

##### 9.1) Cleaning 'industry' column 

In [36]:
# Unique values in industry column
df["industry"].nunique()

1226

In [37]:
# Removing extra whitespace and converting to title case
df["industry"] = df["industry"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()

df["industry"].nunique()

1009

##### 9.2) Cleaning 'job_title' column

In [38]:
# Unique values in job_title column
df["job_title"].nunique()

14430

In [39]:
# Removing extra whitespace and converting to title case
df["job_title"] = df["job_title"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
df["job_title"].nunique()

12181

### 10) Exporting Cleaned Data

In [40]:
df.head(3)

,timestamp,age_group,industry,job_title,annual_salary,other_compensation,currency,country,state_us_only,city,experience_overall,experience_field,education,gender,race
0,2021-04-27 11:02:10,25-34,Education (Higher Education),Research And Instruction Librarian,55000,0.0,USD,United States,[Massachusetts],Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,2021-04-27 11:02:22,25-34,Computing Or Tech,Change & Internal Communications Manager,54600,4000.0,GBP,United Kingdom,[Outside_US],Cambridge,8 - 10 years,5-7 years,College degree,Other or prefer not to answer,White
2,2021-04-27 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,34000,0.0,USD,United States,[Tennessee],Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28211 entries, 0 to 28210
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   timestamp           28211 non-null  datetime64[ns]
 1   age_group           28211 non-null  object        
 2   industry            28211 non-null  object        
 3   job_title           28211 non-null  object        
 4   annual_salary       28211 non-null  int64         
 5   other_compensation  28211 non-null  float64       
 6   currency            28211 non-null  object        
 7   country             28211 non-null  object        
 8   state_us_only       28211 non-null  object        
 9   city                28211 non-null  object        
 10  experience_overall  28211 non-null  object        
 11  experience_field    28211 non-null  object        
 12  education           28211 non-null  object        
 13  gender              28211 non-null  object    

##### 10.1) Exporting cleaned dataframe as csv file

In [42]:
df.to_csv("datasets/manager_salary_survey_cleaned.csv", index=False)

##### 10.2) Exporting columns description as other csv file

In [43]:
df_columns =  pd.DataFrame(list(column_map.items()), columns=["column_description", "column_name"]).set_index("column_name")

In [44]:
df_columns.to_csv("datasets/manager_salary_survey_metadata.csv")

In [45]:
df_columns

,column_description
column_name,
timestamp,Timestamp
age_group,How old are you?
industry,What industry do you work in?
job_title,Job title
job_context,"If your job title needs additional context, pl..."
annual_salary,What is your annual salary? (You'll indicate t...
other_compensation,How much additional monetary compensation do y...
currency,Please indicate the currency
other_currency,"If ""Other,"" please indicate the currency here:"


### -------------------------------------------------- END ---------------------------------------------------